# Runnable Routing

The `router.py` module defines a serializable Runnable that selects another Runnable from a key-to-Runnable mapping.

Each routing input contains a `key` used to select the destination Runnable and an `input` value passed to that Runnable. The router supports synchronous and asynchronous invocation, batching, and streaming.

# RouterInput

`RouterInput` defines the structured input accepted by `RouterRunnable`.

## Bases

- `TypedDict`

## Attributes

1. `key`: Stores the key used to select a Runnable from the router's mapping.
   * **Type:**
     ```python
     key: str
     ```

2. `input`: Stores the value passed to the selected Runnable.
   * **Type:**
     ```python
     input: Any
     ```

# RouterRunnable

`RouterRunnable` selects and executes one Runnable based on `RouterInput["key"]`.

The selected Runnable receives `RouterInput["input"]` as its actual input. A `ValueError` is raised when the requested key does not exist in the router's mapping.

## Bases

- `RunnableSerializable[RouterInput, Output]`

## Attributes

1. `runnables`: Maps routing keys to the Runnables that may be selected.
   * **Type:**
     ```python
     runnables: Mapping[
         str,
         Runnable[
             Any,
             Output
         ]
     ]
     ```

## Configuration

1. `model_config`: Allows arbitrary Python types in the Pydantic model.
   * **Definition:**
     ```python
     model_config = ConfigDict(
         arbitrary_types_allowed=True
     )
     ```

### Properties

1. `config_specs`: Returns the unique configuration specifications exposed by every mapped Runnable.

   Duplicate configuration specifications are removed.

   * **Type:**
     ```python
     config_specs: list[
         ConfigurableFieldSpec
     ]
     ```

### Methods

1. `__init__`: Creates a router from a mapping of keys to Runnables or compatible callables.

   Each supplied value is converted through `coerce_to_runnable`, allowing ordinary callables to be used alongside existing Runnable objects.

   * **Syntax:**
     ```python
     __init__(
         self,
         runnables: Mapping[
             str,
             Runnable[
                 Any,
                 Output
             ]
             | Callable[
                 [Any],
                 Output
             ]
         ] # Mapping of route keys to Runnables or callables
     ) -> None
     ```

2. `is_lc_serializable`: Indicates that `RouterRunnable` supports LangChain serialization.
   * **Syntax:**
     ```python
     @classmethod
     is_lc_serializable(
         cls
     ) -> bool
     ```

3. `get_lc_namespace`: Returns the LangChain serialization namespace.
   * **Syntax:**
     ```python
     @classmethod
     get_lc_namespace(
         cls
     ) -> list[str]
     ```

4. `invoke`: Selects a Runnable and executes it synchronously.

   The router reads `input["key"]`, passes `input["input"]` to the corresponding Runnable, and returns its output.

   A `ValueError` is raised when no Runnable is associated with the supplied key.

   * **Syntax:**
     ```python
     invoke(
         self,
         input: RouterInput, # Routing key and actual Runnable input
         config: RunnableConfig | None = None, # Runtime configuration
         **kwargs: Any # Additional interface arguments
     ) -> Output
     ```

5. `ainvoke`: Selects a Runnable and executes it asynchronously.

   It follows the same routing and unknown-key behaviour as `invoke`.

   * **Syntax:**
     ```python
     async ainvoke(
         self,
         input: RouterInput, # Routing key and actual Runnable input
         config: RunnableConfig | None = None, # Runtime configuration
         **kwargs: Any | None # Additional interface arguments
     ) -> Output
     ```

6. `batch`: Routes and executes multiple inputs synchronously.

   Each input may select a different Runnable. Shared or per-input configurations are normalized through `get_config_list`, and execution is performed through the executor selected from the first normalized configuration.

   When `return_exceptions` is enabled, exceptions raised by individual selected Runnables are returned in their corresponding output positions. Otherwise, they are raised.

   An empty input list returns an empty list. A `ValueError` is raised before execution when one or more routing keys are unknown.

   * **Syntax:**
     ```python
     batch(
         self,
         inputs: list[RouterInput], # Routing inputs to process
         config: RunnableConfig
         | list[RunnableConfig]
         | None = None, # Shared or per-input runtime configuration
         *,
         return_exceptions: bool = False, # Return individual exceptions instead of raising them
         **kwargs: Any | None # Arguments passed to each selected Runnable
     ) -> list[Output]
     ```

7. `abatch`: Routes and executes multiple inputs asynchronously.

   Each input may select a different Runnable. Per-input asynchronous invocations are gathered with the `max_concurrency` value from the first normalized configuration.

   When `return_exceptions` is enabled, exceptions raised by individual selected Runnables are returned in their corresponding output positions. Otherwise, they are raised.

   An empty input list returns an empty list. A `ValueError` is raised before execution when one or more routing keys are unknown.

   * **Syntax:**
     ```python
     async abatch(
         self,
         inputs: list[RouterInput], # Routing inputs to process
         config: RunnableConfig
         | list[RunnableConfig]
         | None = None, # Shared or per-input runtime configuration
         *,
         return_exceptions: bool = False, # Return individual exceptions instead of raising them
         **kwargs: Any | None # Arguments passed to each selected Runnable
     ) -> list[Output]
     ```

8. `stream`: Selects a Runnable and synchronously yields its streamed output.

   The router delegates streaming to the selected Runnable after extracting the actual input value.

   A `ValueError` is raised when no Runnable is associated with the supplied key.

   * **Syntax:**
     ```python
     stream(
         self,
         input: RouterInput, # Routing key and actual Runnable input
         config: RunnableConfig | None = None, # Runtime configuration
         **kwargs: Any | None # Additional interface arguments
     ) -> Iterator[Output]
     ```

9. `astream`: Selects a Runnable and asynchronously yields its streamed output.

   It follows the same routing and unknown-key behaviour as `stream`.

   * **Syntax:**
     ```python
     async astream(
         self,
         input: RouterInput, # Routing key and actual Runnable input
         config: RunnableConfig | None = None, # Runtime configuration
         **kwargs: Any | None # Additional interface arguments
     ) -> AsyncIterator[Output]
     ```

## Routing Behaviour

The router performs the following steps for individual execution:

1. Reads the route key from `RouterInput["key"]`.
2. Reads the destination input from `RouterInput["input"]`.
3. Verifies that the key exists in `runnables`.
4. Delegates execution to the selected Runnable.
5. Returns or yields the selected Runnable's output.

For batch execution, every key is validated before any selected Runnable is executed.